<a href="https://colab.research.google.com/github/alexandresuescun/training/blob/main/examples/tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!sudo rm -rf /content/datasets/
!sudo rm -rf /content/drive/MyDrive/work/checkpoints/relu/
!sudo rm -rf /content/drive/MyDrive/work/checkpoints/relu_in_head/

!mkdir -p /content/datasets
!mkdir -p /content/drive/MyDrive/work/checkpoints/relu
!mkdir -p /content/drive/MyDrive/work/checkpoints/relu_in_head

!cp /content/drive/MyDrive/work/skiers-and-boarders.v3i.yolov8.zip \
     /content/datasets/skiersv1.zip

!unzip -q /content/datasets/skiersv1.zip -d /content/datasets/skiersv1

from ultralytics import YOLO
import torch
from pathlib import Path

def replace_silu_with_relu(module):
    for name, child in module.named_children():
        if isinstance(child, torch.nn.SiLU):
            setattr(module, name, torch.nn.ReLU(inplace=True))
        else:
            replace_silu_with_relu(child)

def replace_silu_with_relu_in_head(module):
    if hasattr(module, 'model') and hasattr(module.model, 'model'):
        target = module.model.model[-1]
    elif hasattr(module, 'model'):
        target = module.model[-1]
    else:
        target = module[-1]

    def recursive_replace(m):
        for name, child in m.named_children():
            if isinstance(child, torch.nn.SiLU):
                setattr(m, name, torch.nn.ReLU(inplace=True))
            else:
                recursive_replace(child)
    recursive_replace(target)

def make_checkpoint_callback(drive_save_dir, replace_fn):
    save_dir = Path(drive_save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    prev_best_mtime = [None]

    def on_train_start(trainer):
        replace_fn(trainer.model)  # re-apply after Ultralytics setup
        print("✅ Architecture replacement confirmed")

    def on_fit_epoch_end(trainer):
        epoch = trainer.epoch

        # ── Verify ReLU replacement held ─────────────────────────────────────────
        silu_found = [name for name, m in trainer.model.named_modules() if isinstance(m, torch.nn.SiLU)]
        relu_count = sum(1 for _, m in trainer.model.named_modules() if isinstance(m, torch.nn.ReLU))
        if silu_found:
            print(f"⚠️ Epoch {epoch}: SiLU still present in {silu_found}")
        else:
            print(f"✅ Epoch {epoch}: no SiLU found, {relu_count} ReLU ops active")

        # ── Save checkpoints ──────────────────────────────────────────────────────
        torch.save(trainer.model.state_dict(), save_dir / f"last_epoch{epoch:03d}.pt")
        best = Path(trainer.best)
        if best.exists():
            current_mtime = best.stat().st_mtime
            if current_mtime != prev_best_mtime[0]:
                torch.save(trainer.model.state_dict(), save_dir / f"best_epoch{epoch:03d}.pt")
                prev_best_mtime[0] = current_mtime
                print(f"✅ New best saved → best_epoch{epoch:03d}.pt")

    return on_train_start, on_fit_epoch_end



model = YOLO('yolov8n.pt')
replace_silu_with_relu(model)

on_train_start, on_fit_epoch_end = make_checkpoint_callback(
    "/content/drive/MyDrive/work/checkpoints/relu",
    replace_silu_with_relu
)
model.add_callback("on_train_start", on_train_start)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

model.train(data="/content/datasets/skiersv1/data.yaml", epochs=300)

torch.save(
    model.model.state_dict(),
    "/content/drive/MyDrive/work/checkpoints/relu/final.pt"
)







model = YOLO('yolov8n.pt')
replace_silu_with_relu_in_head(model)

on_train_start, on_fit_epoch_end = make_checkpoint_callback(
    "/content/drive/MyDrive/work/checkpoints/relu_in_head",
    replace_silu_with_relu_in_head
)
model.add_callback("on_train_start", on_train_start)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

model.train(data="/content/datasets/skiersv1/data.yaml", epochs=300)

torch.save(
    model.model.state_dict(),
    "/content/drive/MyDrive/work/checkpoints/relu_in_head/final.pt"
)




Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Setup complete ✅ (48 CPUs, 176.9 GB RAM, 48.5/112.6 GB disk)
Mounted at /content/drive
replace /content/datasets/skiersv1/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/skiersv1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015

In [4]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!sudo rm -rf /content/datasets/
!sudo rm -rf /content/drive/MyDrive/work/checkpoints/all_silu/

!mkdir -p /content/datasets
!mkdir -p /content/drive/MyDrive/work/checkpoints/all_silu

!cp /content/drive/MyDrive/work/skiers-and-boarders.v3i.yolov8.zip \
     /content/datasets/skiersv1.zip

!unzip -q /content/datasets/skiersv1.zip -d /content/datasets/skiersv1

from ultralytics import YOLO
import torch
from pathlib import Path

def make_checkpoint_callback(drive_save_dir):
    save_dir = Path(drive_save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    prev_best_mtime = [None]

    def on_fit_epoch_end(trainer):
      epoch = trainer.epoch

      torch.save(torch.load(trainer.last, map_location='cpu', weights_only=False), save_dir / f"last_epoch{epoch:03d}.pt")

      best = Path(trainer.best)
      if best.exists():
          current_mtime = best.stat().st_mtime
          if current_mtime != prev_best_mtime[0]:
              torch.save(torch.load(trainer.best, map_location='cpu', weights_only=False), save_dir / f"best_epoch{epoch:03d}.pt")
              prev_best_mtime[0] = current_mtime
              print(f"✅ New best saved → best_epoch{epoch:03d}.pt")

    return on_fit_epoch_end

model = YOLO('yolov8n.pt')

on_fit_epoch_end = make_checkpoint_callback(
    "/content/drive/MyDrive/work/checkpoints/all_silu"
)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

model.train(data="/content/datasets/skiersv1/data.yaml", epochs=300)

torch.save(
    model.model.state_dict(),
    "/content/drive/MyDrive/work/checkpoints/all_silu/final.pt"
)


Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Setup complete ✅ (48 CPUs, 176.9 GB RAM, 54.2/112.6 GB disk)
Mounted at /content/drive
replace /content/datasets/skiersv1/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/skiersv1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015